# Silver layer

## Product additional information

Table to store additional information about products:
* Id
* Category
* Subcategory
* If maintenance is needed

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

## Read table from bronze layer

In [0]:
df = spark.read.table("db_project.bronze.erp_px_cat_g1v2")
df.display()

## Correct Strings

Correct Strings

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        display(field.dataType)
        df = df.withColumn(field.name, F.trim(col(field.name)))

## Check nulls

In [0]:
test_nulls = df.where(
    df.id.isNull() |
    df.cat.isNull() |
    df.subcat.isNull() |
    df.MAINTENANCE.isNull()
)
test_nulls.display()

## Check Maintenance

See if there are no unexpected values

In [0]:
test_maintenance = df.select(df.maintenance).distinct()
test_maintenance.display()

# Write table silver.erp_px_cat_g1v2

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("ovrwriteSchema", "true").format("delta").saveAsTable("db_project.silver.erp_px_cat_g1v2")